# 01 — Dates, Calendars & Day Counters

This notebook covers the foundational building blocks of quantitative finance:  
**date arithmetic**, **holiday calendars**, **business-day conventions**, **day-count fractions**, and **schedule generation**.

We compare every operation side-by-side between **QuantLib-SWIG** (C++ via Python bindings) and **ql-jax** (pure JAX).

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────
BACKEND = "cpu"  # switch to "gpu" if available

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..") if not os.getcwd().endswith("ql-jax") else ".")
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))

from notebooks._common import setup_backend, load_quantlib, compare, compare_table, timed_ms
jax = setup_backend(BACKEND)
QL = load_quantlib()

---
## 1. Date Construction & Arithmetic

A `Date` encodes a specific calendar day. Both libraries support construction from (day, month, year) and serial numbers, plus arithmetic with periods.

In [ ]:
# ── QuantLib ────────────────────────────────────────────────────────────────
ql_date = QL.Date(15, 6, 2024)
print(f"QL  Date          : {ql_date}")
print(f"QL  serial        : {ql_date.serialNumber()}")
print(f"QL  day/month/year: {ql_date.dayOfMonth()}/{ql_date.month()}/{ql_date.year()}")
print(f"QL  weekday       : {ql_date.weekday()}")

# Arithmetic
ql_plus_30d  = ql_date + QL.Period(30, QL.Days)
ql_plus_3m   = ql_date + QL.Period(3, QL.Months)
ql_plus_1y   = ql_date + QL.Period(1, QL.Years)
print(f"QL  +30d          : {ql_plus_30d}")
print(f"QL  +3M           : {ql_plus_3m}")
print(f"QL  +1Y           : {ql_plus_1y}")

In [ ]:
# ── ql-jax ──────────────────────────────────────────────────────────────────
from ql_jax.time.date import Date, Period
from ql_jax._util.types import TimeUnit

jax_date = Date(15, 6, 2024)
print(f"JAX Date          : {jax_date}")
print(f"JAX serial        : {jax_date.serial}")
print(f"JAX day/month/year: {jax_date.day}/{jax_date.month}/{jax_date.year}")

# Arithmetic
jax_plus_30d = jax_date.advance(30, TimeUnit.Days)
jax_plus_3m  = jax_date.advance(3, TimeUnit.Months)
jax_plus_1y  = jax_date.advance(1, TimeUnit.Years)
print(f"JAX +30d          : {jax_plus_30d}")
print(f"JAX +3M           : {jax_plus_3m}")
print(f"JAX +1Y           : {jax_plus_1y}")

In [ ]:
# ── Comparison ──────────────────────────────────────────────────────────────
compare_table([
    ("Serial number",     ql_date.serialNumber(),          jax_date.serial),
    ("+30 days (serial)", ql_plus_30d.serialNumber(),      jax_plus_30d.serial),
    ("+3 months (serial)", ql_plus_3m.serialNumber(),      jax_plus_3m.serial),
    ("+1 year (serial)",  ql_plus_1y.serialNumber(),       jax_plus_1y.serial),
], title="Date Arithmetic")

---
## 2. Holiday Calendars

Financial calendars encode market holidays. Both libraries ship a wide collection (TARGET, US, UK, Japan, etc.).

In [ ]:
# ── QuantLib ────────────────────────────────────────────────────────────────
ql_target = QL.TARGET()
ql_us     = QL.UnitedStates(QL.UnitedStates.GovernmentBond)

# Check a few dates
test_dates_ql = [
    QL.Date(1, 1, 2024),   # New Year
    QL.Date(15, 1, 2024),  # MLK Day (US)
    QL.Date(29, 3, 2024),  # Good Friday
    QL.Date(1, 5, 2024),   # Labour Day (TARGET)
    QL.Date(4, 7, 2024),   # Independence Day (US)
    QL.Date(17, 6, 2024),  # Regular Monday
]

print("QuantLib calendar checks:")
for d in test_dates_ql:
    print(f"  {d}  TARGET={ql_target.isBusinessDay(d)}  US={ql_us.isBusinessDay(d)}")

In [ ]:
# ── ql-jax ──────────────────────────────────────────────────────────────────
from ql_jax.time.calendar import TARGET, UnitedStates, NullCalendar

jax_target = TARGET()
jax_us     = UnitedStates()

test_dates_jax = [
    Date(1, 1, 2024),
    Date(15, 1, 2024),
    Date(29, 3, 2024),
    Date(1, 5, 2024),
    Date(4, 7, 2024),
    Date(17, 6, 2024),
]

print("ql-jax calendar checks:")
for d in test_dates_jax:
    print(f"  {d}  TARGET={jax_target.is_business_day(d)}  US={jax_us.is_business_day(d)}")

In [ ]:
# ── Comparison ──────────────────────────────────────────────────────────────
rows = []
for dql, djax in zip(test_dates_ql, test_dates_jax):
    rows.append((f"TARGET isBusinessDay({dql})",
                 int(ql_target.isBusinessDay(dql)),
                 int(jax_target.is_business_day(djax))))
    rows.append((f"US isBusinessDay({dql})",
                 int(ql_us.isBusinessDay(dql)),
                 int(jax_us.is_business_day(djax))))

compare_table(rows, title="Calendar Business-Day Checks")

---
## 3. Business-Day Adjustment & Advance

When a payment falls on a holiday, conventions specify how to adjust:
- **Following**: next business day
- **Modified Following**: next business day unless it crosses a month boundary (then preceding)
- **Preceding**: previous business day

In [ ]:
from ql_jax._util.types import BusinessDayConvention

# Saturday 2024-06-15 → adjust under different conventions
ql_sat = QL.Date(15, 6, 2024)
jax_sat = Date(15, 6, 2024)

conventions = [
    ("Following",          QL.Following,          BusinessDayConvention.Following),
    ("ModifiedFollowing",  QL.ModifiedFollowing,  BusinessDayConvention.ModifiedFollowing),
    ("Preceding",          QL.Preceding,          BusinessDayConvention.Preceding),
]

rows = []
for label, ql_conv, jax_conv in conventions:
    ql_adj  = ql_target.adjust(ql_sat, ql_conv).serialNumber()
    jax_adj = jax_target.adjust(jax_sat, jax_conv).serial
    rows.append((f"adjust({ql_sat}, {label})", ql_adj, jax_adj))

compare_table(rows, title="Business-Day Adjustment")

In [ ]:
# Calendar.advance — move forward by a period, respecting holidays
rows = []
ql_start = QL.Date(28, 3, 2024)   # Thursday before Good Friday
jax_start = Date(28, 3, 2024)

for n_days in [1, 2, 5, 10]:
    ql_adv  = ql_target.advance(ql_start, n_days, QL.Days).serialNumber()
    jax_adv = jax_target.advance(jax_start, n_days, TimeUnit.Days).serial
    rows.append((f"advance(+{n_days}D, TARGET)", ql_adv, jax_adv))

compare_table(rows, title="Calendar Advance (Business Days)")

---
## 4. Business Days Between Two Dates

In [ ]:
ql_d1, ql_d2 = QL.Date(1, 1, 2024), QL.Date(31, 12, 2024)
jax_d1, jax_d2 = Date(1, 1, 2024), Date(31, 12, 2024)

ql_bdays_target = ql_target.businessDaysBetween(ql_d1, ql_d2)
ql_bdays_us     = ql_us.businessDaysBetween(ql_d1, ql_d2)

jax_bdays_target = jax_target.business_days_between(jax_d1, jax_d2)
jax_bdays_us     = jax_us.business_days_between(jax_d1, jax_d2)

compare_table([
    ("BDays 2024 (TARGET)", ql_bdays_target, jax_bdays_target),
    ("BDays 2024 (US Govt)", ql_bdays_us, jax_bdays_us),
], title="Business Days Between")

---
## 5. Day-Count Conventions

Day-count conventions determine how to compute the **year fraction** between two dates.  
Common conventions:

| Convention | Formula |
|---|---|
| Actual/365 (Fixed) | $\tau = \frac{\text{actual days}}{365}$ |
| Actual/360 | $\tau = \frac{\text{actual days}}{360}$ |
| 30/360 | $\tau = \frac{360(Y_2-Y_1) + 30(M_2-M_1) + (D_2-D_1)}{360}$ |
| Actual/Actual (ISDA) | Actual days in each year, weighted |
| Actual/Actual (ISMA) | Uses coupon period as reference |

In [ ]:
from ql_jax.time.daycounter import year_fraction, DayCountConvention

d1_ql, d2_ql = QL.Date(15, 3, 2024), QL.Date(15, 9, 2024)
d1_jx, d2_jx = Date(15, 3, 2024), Date(15, 9, 2024)

dc_pairs = [
    ("Actual/365 (Fixed)", QL.Actual365Fixed(),   DayCountConvention.Actual365Fixed),
    ("Actual/360",         QL.Actual360(),         DayCountConvention.Actual360),
    ("30/360",             QL.Thirty360(QL.Thirty360.BondBasis), DayCountConvention.Thirty360),
    ("Actual/Actual ISDA", QL.ActualActual(QL.ActualActual.ISDA), DayCountConvention.ActualActualISDA),
]

rows = []
for label, ql_dc, jax_dc in dc_pairs:
    ql_yf  = ql_dc.yearFraction(d1_ql, d2_ql)
    jax_yf = float(year_fraction(d1_jx, d2_jx, jax_dc))
    rows.append((label, ql_yf, jax_yf))

compare_table(rows, title="Year Fractions (2024-03-15 → 2024-09-15)")

In [ ]:
# Year fraction across a leap year boundary
d1_ql2, d2_ql2 = QL.Date(1, 1, 2024), QL.Date(1, 1, 2025)
d1_jx2, d2_jx2 = Date(1, 1, 2024), Date(1, 1, 2025)

rows2 = []
for label, ql_dc, jax_dc in dc_pairs:
    ql_yf  = ql_dc.yearFraction(d1_ql2, d2_ql2)
    jax_yf = float(year_fraction(d1_jx2, d2_jx2, jax_dc))
    rows2.append((label, ql_yf, jax_yf))

compare_table(rows2, title="Year Fractions: Full Leap Year 2024")

---
## 6. Schedule Generation

A **schedule** is a sequence of dates used for coupon payments, resets, etc.  
Key parameters: start, end, frequency, calendar, business-day convention, date-generation rule.

In [ ]:
from ql_jax.time.schedule import MakeSchedule
from ql_jax._util.types import Frequency, DateGeneration

# ── QuantLib ────────────────────────────────────────────────────────────────
ql_sched = QL.Schedule(
    QL.Date(15, 1, 2024), QL.Date(15, 1, 2029),
    QL.Period(QL.Semiannual),
    QL.TARGET(),
    QL.ModifiedFollowing, QL.ModifiedFollowing,
    QL.DateGeneration.Backward, False,
)

print(f"QuantLib schedule ({len(ql_sched)} dates):")
for i in range(len(ql_sched)):
    print(f"  {i:2d}: {ql_sched[i]}")

In [ ]:
# ── ql-jax ──────────────────────────────────────────────────────────────────
jax_sched = (MakeSchedule()
    .from_date(Date(15, 1, 2024))
    .to_date(Date(15, 1, 2029))
    .with_frequency(Frequency.Semiannual)
    .with_calendar(TARGET())
    .with_convention(BusinessDayConvention.ModifiedFollowing)
    .backwards()
    .build())

print(f"ql-jax schedule ({len(jax_sched)} dates):")
for i, d in enumerate(jax_sched.dates):
    print(f"  {i:2d}: {d}")

In [ ]:
# ── Comparison ──────────────────────────────────────────────────────────────
rows = []
rows.append(("Number of dates", len(ql_sched), len(jax_sched.dates)))
n = min(len(ql_sched), len(jax_sched.dates))
for i in range(n):
    rows.append((f"Date[{i}] serial", ql_sched[i].serialNumber(), jax_sched.dates[i].serial))

compare_table(rows, title="Schedule Comparison (5Y Semi-Annual, TARGET, ModFol, Backward)")

---
## 7. Performance: Generating 10,000 Schedules

Let's benchmark generating many schedules — a task common in portfolio construction.

In [ ]:
from notebooks._common import timed_ms, plot_speedup

N = 1000

def gen_ql_schedules():
    for i in range(N):
        s = QL.Schedule(
            QL.Date(15, 1, 2024), QL.Date(15, 1, 2029),
            QL.Period(QL.Semiannual), QL.TARGET(),
            QL.ModifiedFollowing, QL.ModifiedFollowing,
            QL.DateGeneration.Backward, False)
    return s

def gen_jax_schedules():
    for i in range(N):
        s = (MakeSchedule()
            .from_date(Date(15, 1, 2024))
            .to_date(Date(15, 1, 2029))
            .with_frequency(Frequency.Semiannual)
            .with_calendar(TARGET())
            .with_convention(BusinessDayConvention.ModifiedFollowing)
            .backwards()
            .build())
    return s

ql_ms, _  = timed_ms(gen_ql_schedules, warmup=1, runs=3)
jax_ms, _ = timed_ms(gen_jax_schedules, warmup=1, runs=3)

print(f"\n{N} schedules:")
print(f"  QuantLib : {ql_ms:8.1f} ms")
print(f"  ql-jax   : {jax_ms:8.1f} ms")
print(f"  Ratio    : {ql_ms/jax_ms:.2f}×" if jax_ms > 0 else "")

---
## 8. Holiday Lists & Calendar Combinations

In [ ]:
# Holiday list for 2024
ql_holidays = QL.TARGET().holidayList(
    QL.Date(1, 1, 2024), QL.Date(31, 12, 2024))
print(f"TARGET holidays 2024 ({len(ql_holidays)}):")
for h in ql_holidays:
    print(f"  {h}")

jax_holidays = jax_target.holiday_list(Date(1, 1, 2024), Date(31, 12, 2024))
print(f"\nql-jax TARGET holidays 2024 ({len(jax_holidays)}):")
for h in jax_holidays:
    print(f"  {h}")

compare("Number of TARGET holidays 2024", len(ql_holidays), len(jax_holidays))

In [ ]:
# Joint calendar: TARGET + US
from ql_jax.time.calendar import JointCalendar

ql_joint = QL.JointCalendar(QL.TARGET(), QL.UnitedStates(QL.UnitedStates.GovernmentBond))
jax_joint = JointCalendar(TARGET(), UnitedStates())

d_test = QL.Date(1, 5, 2024)  # Labour Day (TARGET holiday, not US)
j_test = Date(1, 5, 2024)

compare_table([
    ("Joint isBusinessDay(2024-05-01)",
     int(ql_joint.isBusinessDay(d_test)),
     int(jax_joint.is_business_day(j_test))),
    ("Joint bdays 2024",
     ql_joint.businessDaysBetween(QL.Date(1,1,2024), QL.Date(31,12,2024)),
     jax_joint.business_days_between(Date(1,1,2024), Date(31,12,2024))),
], title="Joint Calendar (TARGET ∪ US)")

---
## 9. End-of-Month Rules

In [ ]:
# End-of-month static functions
eom_tests = [
    ("Feb 2024 (leap)",   Date(29, 2, 2024)),
    ("Feb 2023 (no leap)",Date(28, 2, 2023)),
    ("Jun 2024",          Date(30, 6, 2024)),
]

rows = []
for label, d in eom_tests:
    ql_d = QL.Date(d.day, d.month, d.year)
    rows.append((f"isEndOfMonth({label})",
                 int(QL.Date.isEndOfMonth(ql_d)),
                 int(Date.is_end_of_month(d))))

# Also check end_of_month
ql_eom = QL.Date.endOfMonth(QL.Date(15, 2, 2024))
jax_eom = Date.end_of_month(Date(15, 2, 2024))
rows.append(("endOfMonth(2024-02-15)", ql_eom.serialNumber(), jax_eom.serial))

compare_table(rows, title="End-of-Month")

---
## 10. Exercises

1. **New calendar**: Create a `JointCalendar` for Tokyo + London. Count the business days in 2024 and compare against QuantLib.
2. **Day-count comparison**: Compute `Actual/Actual (ISMA)` year fractions for a semi-annual bond coupon period and compare.
3. **Schedule variants**: Build quarterly schedules with Forward vs Backward date generation and verify they produce the same payment dates.

---
*End of Notebook 01*